# Modbus/TCP PCAP Feature Extraction for Anomaly Detection

This notebook extracts Modbus/TCP fields from PCAP files using tshark and exports them to CSV for ML-based anomaly detection.

In [1]:
# Requires tshark (Wireshark CLI) to be installed and in PATH
# Download from: https://www.wireshark.org/download.html

In [2]:
import subprocess
import json
import pandas as pd
from datetime import datetime
from pathlib import Path
import csv
import os

In [ ]:
def extract_modbus_features(pcap_path: str) -> list[dict]:
    """
    Extract Modbus/TCP fields from a PCAP file using tshark.
    
    Args:
        pcap_path: Path to the PCAP file
        
    Returns:
        List of dictionaries containing extracted features
    """
    # Define fields to extract (mbtcp = Modbus/TCP header, modbus = PDU)
    fields = [
        "frame.number",
        "frame.time_epoch",
        "frame.len",
        "_ws.col.protocol",
        "ip.src",
        "ip.dst",
        "tcp.srcport",
        "tcp.dstport",
        "tcp.len",
        "mbtcp.trans_id",
        "mbtcp.prot_id",
        "mbtcp.len",
        "mbtcp.unit_id",
        "modbus.func_code",
        "modbus.reference_num",
        "modbus.word_cnt",
        "modbus.bit_cnt",
        "modbus.byte_cnt",
        "modbus.exception_code",
    ]
    
    # Build tshark command
    cmd = [
        "tshark",
        "-r", pcap_path,  
        "-T", "fields",
        # "-Y", "modbus",  # Uncomment to filter only Modbus packets
        "-E", "header=y",
        "-E", "separator=|",
        "-E", "quote=d",
    ]

    
    # Add field arguments
    for field in fields:
        cmd.extend(["-e", field])
    
    # Run tshark
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        raise RuntimeError(f"tshark failed: {result.stderr}")
    
    # Parse output
    lines = result.stdout.strip().split("\n")
    if len(lines) < 2:
        return []
    
    headers = lines[0].split("|")
    records = []
    
    def parse_int(val):
        """Parse int, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return int(val)
        except ValueError:
            return None
    
    def parse_float(val):
        """Parse float, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        try:
            return float(val)
        except ValueError:
            return None
    
    def parse_str(val):
        """Parse string, taking first value if comma-separated."""
        if not val:
            return None
        val = val.strip('"')
        if ',' in val:
            val = val.split(',')[0]
        return val or None
    
    # Process each line
    for line in lines[1:]:
        values = line.split("|")
        row = dict(zip(headers, values))
        
        timestamp = parse_float(row.get('frame.time_epoch'))
        
        record = {
            'packet_number': parse_int(row.get('frame.number')),
            'timestamp': timestamp,
            'datetime': datetime.fromtimestamp(timestamp).isoformat() if timestamp else None,
            'protocols': parse_str(row.get('_ws.col.protocol')),
            'src_ip': parse_str(row.get('ip.src')),
            'dst_ip': parse_str(row.get('ip.dst')),
            'src_port': parse_int(row.get('tcp.srcport')),
            'dst_port': parse_int(row.get('tcp.dstport')),
            'tcp_len': parse_int(row.get('tcp.len')),
            'transaction_id': parse_int(row.get('mbtcp.trans_id')),
            'protocol_id': parse_int(row.get('mbtcp.prot_id')),
            'modbus_length': parse_int(row.get('mbtcp.len')),
            'unit_id': parse_int(row.get('mbtcp.unit_id')),
            'function_code': parse_int(row.get('modbus.func_code')),
            'reference_num': parse_int(row.get('modbus.reference_num')),
            'word_count': parse_int(row.get('modbus.word_cnt')),
            'bit_count': parse_int(row.get('modbus.bit_cnt')),
            'byte_count': parse_int(row.get('modbus.byte_cnt')),
            'exception_code': parse_int(row.get('modbus.exception_code')),
            'pkt_len': parse_int(row.get('frame.len')),
        }
        
        # Derive additional features
        record['is_request'] = 1 if record['dst_port'] == 502 else 0
        record['is_response'] = 1 if record['src_port'] == 502 else 0
        record['is_exception'] = 1 if record['exception_code'] is not None else 0
        
        records.append(record)
    
    return records

In [4]:
# Modbus function code reference for labeling
MODBUS_FUNCTION_CODES = {
    1: 'Read Coils',
    2: 'Read Discrete Inputs',
    3: 'Read Holding Registers',
    4: 'Read Input Registers',
    5: 'Write Single Coil',
    6: 'Write Single Register',
    7: 'Read Exception Status',
    8: 'Diagnostics',
    15: 'Write Multiple Coils',
    16: 'Write Multiple Registers',
    22: 'Mask Write Register',
    23: 'Read/Write Multiple Registers',
    43: 'Read Device Identification',
}

def add_function_name(df: pd.DataFrame) -> pd.DataFrame:
    """Add human-readable function code names."""
    df['function_name'] = df['function_code'].map(MODBUS_FUNCTION_CODES).fillna('Unknown')
    return df

In [5]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add time-based features useful for anomaly detection.
    """
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    # Inter-arrival time
    df['inter_arrival_time'] = df['timestamp'].diff()
    
    # Time since first packet
    df['time_from_start'] = df['timestamp'] - df['timestamp'].iloc[0]
    
    # Rolling statistics (last 10 packets)
    df['rolling_iat_mean'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).mean()
    df['rolling_iat_std'] = df['inter_arrival_time'].rolling(window=10, min_periods=1).std()
    
    return df

In [6]:
def add_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add flow-based features for each unique connection pair.
    """
    # Create flow identifier
    df['flow_id'] = df.apply(
        lambda x: f"{min(str(x['src_ip']), str(x['dst_ip']))}_{max(str(x['src_ip']), str(x['dst_ip']))}",
        axis=1
    )
    
    # Packet count per flow
    df['flow_pkt_count'] = df.groupby('flow_id').cumcount() + 1
    
    # Function code diversity per flow (rolling)
    df['unique_func_codes'] = df.groupby('flow_id')['function_code'].transform(
        lambda x: x.expanding().apply(lambda y: y.nunique())
    )
    
    return df

## Usage Example

In [7]:
# === CONFIGURE YOUR PCAP FILE PATH HERE ===
relative_path = "data\\raw\\captures1_v2\\mitm\\eth2dump-mitm-change-5m-1h_1.pcap"
absolute_path = (Path("..") / relative_path).resolve()

print(f"Using PCAP file at: {absolute_path}")

PCAP_PATH = absolute_path  # Update this path
OUTPUT_CSV = "modbus_features.csv"

Using PCAP file at: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\mitm\eth2dump-mitm-change-5m-1h_1.pcap


In [8]:
# Extract raw Modbus features
print(f"Processing: {PCAP_PATH}")
records = extract_modbus_features(PCAP_PATH)
print(f"Extracted {len(records)} Modbus packets")

Processing: C:\Users\jorel\OneDrive\Documents\CODE STUFF\DOE\BNL\foundational_model_for_energy_security\data\raw\captures1_v2\mitm\eth2dump-mitm-change-5m-1h_1.pcap
Extracted 69440 Modbus packets


In [9]:
# Convert to DataFrame and add derived features
df = pd.DataFrame(records)

if len(df) > 0:
    df = add_function_name(df)
    df = add_temporal_features(df)
    df = add_flow_features(df)
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
else:
    print("No Modbus packets found in PCAP!")


DataFrame shape: (69440, 31)

Columns: ['packet_number', 'timestamp', 'datetime', 'protocols', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'tcp_len', 'transaction_id', 'protocol_id', 'modbus_length', 'unit_id', 'function_code', 'reference_num', 'word_count', 'bit_count', 'byte_count', 'exception_code', 'pkt_len', 'is_request', 'is_response', 'is_exception', 'function_name', 'inter_arrival_time', 'time_from_start', 'rolling_iat_mean', 'rolling_iat_std', 'flow_id', 'flow_pkt_count', 'unique_func_codes']


In [10]:
# Preview the data
df.head(10)

,packet_number,timestamp,datetime,protocols,src_ip,dst_ip,src_port,dst_port,tcp_len,transaction_id,...,is_response,is_exception,function_name,inter_arrival_time,time_from_start,rolling_iat_mean,rolling_iat_std,flow_id,flow_pkt_count,unique_func_codes
0,1,1.535066e+09,2018-08-23T19:15:55.127986,TCP,172.27.224.250,172.27.224.251,502.0,50834.0,0.0,NaN,...,1,0,Unknown,NaN,0.000000,NaN,NaN,172.27.224.250_172.27.224.251,1,NaN
1,2,1.535066e+09,2018-08-23T19:15:55.134133,Modbus/TCP,172.27.224.70,172.27.224.250,49499.0,502.0,12.0,0.0,...,0,0,Read Holding Registers,0.006147,0.006147,0.006147,NaN,172.27.224.250_172.27.224.70,1,1.0
2,3,1.535066e+09,2018-08-23T19:15:55.139886,Modbus/TCP,172.27.224.250,172.27.224.70,502.0,49499.0,31.0,0.0,...,1,0,Read Holding Registers,0.005753,0.011900,0.005950,0.000279,172.27.224.250_172.27.224.70,2,1.0
3,4,1.535066e+09,2018-08-23T19:15:55.351995,TCP,172.27.224.70,172.27.224.250,49499.0,502.0,0.0,NaN,...,0,0,Unknown,0.212109,0.224009,0.074670,0.119026,172.27.224.250_172.27.224.70,3,1.0
4,5,1.535066e+09,2018-08-23T19:15:55.446305,Modbus/TCP,172.27.224.70,172.27.224.250,49499.0,502.0,12.0,0.0,...,0,0,Read Holding Registers,0.094310,0.318319,0.079580,0.097679,172.27.224.250_172.27.224.70,4,1.0
5,6,1.535066e+09,2018-08-23T19:15:55.449236,Modbus/TCP,172.27.224.250,172.27.224.70,502.0,49499.0,31.0,0.0,...,1,0,Read Holding Registers,0.002931,0.321250,0.064250,0.091274,172.27.224.250_172.27.224.70,5,1.0
6,7,1.535066e+09,2018-08-23T19:15:55.663990,TCP,172.27.224.70,172.27.224.250,49499.0,502.0,0.0,NaN,...,0,0,Unknown,0.214754,0.536004,0.089334,0.102176,172.27.224.250_172.27.224.70,6,1.0
7,8,1.535066e+09,2018-08-23T19:15:55.758230,Modbus/TCP,172.27.224.70,172.27.224.250,49499.0,502.0,12.0,0.0,...,0,0,Read Holding Registers,0.094240,0.630244,0.090035,0.093292,172.27.224.250_172.27.224.70,7,1.0
8,9,1.535066e+09,2018-08-23T19:15:55.769911,Modbus/TCP,172.27.224.250,172.27.224.70,502.0,49499.0,31.0,0.0,...,1,0,Read Holding Registers,0.011681,0.641925,0.080241,0.090706,172.27.224.250_172.27.224.70,8,1.0
9,10,1.535066e+09,2018-08-23T19:15:55.975978,TCP,172.27.224.70,172.27.224.250,49499.0,502.0,0.0,NaN,...,0,0,Unknown,0.206067,0.847992,0.094221,0.094648,172.27.224.250_172.27.224.70,9,1.0


In [11]:
# Summary statistics
print("Function Code Distribution:")
print(df['function_name'].value_counts())
print("\nBasic Statistics:")
df[['pkt_len', 'inter_arrival_time', 'modbus_length']].describe()

Function Code Distribution:
function_name
Unknown                   46157
Read Holding Registers    22041
Write Single Register      1242
Name: count, dtype: int64

Basic Statistics:


,pkt_len,inter_arrival_time,modbus_length
count,69440.000000,6.943900e+04,23283.000000
mean,66.247825,5.183777e-02,15.004252
std,19.957550,8.181739e-02,9.487260
min,60.000000,9.536743e-07,6.000000
25%,60.000000,2.541542e-04,6.000000
50%,60.000000,8.573055e-03,6.000000
75%,66.000000,9.422207e-02,25.000000
max,1059.000000,1.428620e+00,25.000000


In [12]:
# Export to CSV
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to: {OUTPUT_CSV}")


Saved to: modbus_features.csv


## Feature Summary for Anomaly Detection

| Feature Category | Fields | Use Case |
|-----------------|--------|----------|
| **Protocol** | `function_code`, `unit_id`, `transaction_id` | Detect unauthorized commands |
| **Payload** | `reference_num`, `word_count`, `byte_count`, `reg_value` | Detect data manipulation |
| **Temporal** | `inter_arrival_time`, `rolling_iat_*` | Detect timing anomalies, DoS |
| **Flow** | `flow_pkt_count`, `unique_func_codes` | Detect reconnaissance, scanning |
| **Error** | `is_exception`, `exception_code` | Detect probing, fuzzing |